# Pillar 4 — Benchmarking Utility (Figures 7, 8, 9)

**Figure 7 — Sub-pillar 4a, cross-dataset difficulty**
- **a** Per-model performance across all five datasets, MCQ and open-ended
- **b** Datasets ranked by aggregate difficulty

**Figure 8 — Sub-pillar 4b, resolution sensitivity** (PathOPEN only)

**Figure 9 — Sub-pillar 4c, organ stratification** (PathOPEN only)

## The claim needs narrowing, and the figure should show why

§2.4 claims PathOPEN is *"a harder, more discriminating benchmark for current VLMs than
existing patch-level pathology datasets."* Corrected for chance, it is not:

| dataset | mean accuracy | chance | above chance |
|---|---|---|---|
| PathMMU | 50.1% | 24.3% | 25.7 |
| **PathOPEN** | 42.4% | 20.0% | **22.4** |
| PatchVQA | 41.4% | 20.9% | 20.5 |

PathOPEN sits in the middle. Raw accuracy makes it look harder than PathMMU, but that is
mostly the 5-option format — chance is 20% rather than 24.3%. Panel **b** plots both raw
and chance-corrected difficulty side by side so this is visible rather than buried, and
so a reviewer who computes it themselves finds the paper already said it.

**What Pillar 4 can claim** is the second half of §2.4: PathOPEN's structural features
*"enable evaluation analyses unsupported by any comparator dataset."* Figures 8 and 9 are
that claim, and it holds — neither PathMMU nor PatchVQA carries magnification or organ
metadata at all, so the stratified analyses are only possible on PathOPEN.

## Sample-size honesty

Both stratified figures fall below the paper's own n≥30 flag threshold in most bins:
3 of 6 magnification bins, and 10 of 11 organ systems. Every bin is annotated with its
*n*, reddened when below threshold. A heatmap that hid this would present 13-item cells
as if they were as solid as 44-item ones.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("."))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import nature_style as ns

ns.apply_style()

ZEROSHOT = os.path.join("..", "data_evaluation", "vlm_zeroshot", "analysis")

MCQ_DATASETS = ["PathOPEN", "PathMMU", "PatchVQA"]
OE_DATASETS = ["PathOPEN_OE", "PathVQA_OE", "QuiltVQA_OE"]
CE_DATASETS = ["PathOPEN_CE", "PathVQA_CE", "QuiltVQA_CE"]
MIN_BIN_N = 30   # the paper's own threshold for flagging thin strata

accuracy = pd.read_csv(os.path.join(ZEROSHOT, "mcq_accuracy.csv"))
open_ended = pd.read_csv(os.path.join(ZEROSHOT, "open_ended_metrics.csv"))
close_ended = pd.read_csv(os.path.join(ZEROSHOT, "close_ended_metrics.csv"))
magnification = pd.read_csv(os.path.join(ZEROSHOT, "mcq_by_magnification.csv"))
organ_system = pd.read_csv(os.path.join(ZEROSHOT, "mcq_by_organ_system.csv"))

full = accuracy[accuracy.condition == "full"]
print(f"{len(full)} model x dataset MCQ rows | {len(open_ended)} open-ended rows")

## Figure 7 — cross-dataset difficulty

Five datasets, three metric families. §2.4's own reviewer-concern says cross-format
metrics must be read **within format**, never blended — so MCQ accuracy, close-ended
accuracy and open-ended BERTScore are three separate panels rather than one composite
score.

In [ ]:
# Chance-corrected difficulty: the number the "harder benchmark" claim rests on.
difficulty = (full.groupby("dataset")
              .agg(accuracy=("accuracy", "mean"), chance=("chance", "mean"))
              .assign(above_chance=lambda d: d.accuracy - d.chance)
              .sort_values("above_chance"))
print(difficulty.round(2).to_string())
print()
# Discrimination: does the dataset separate strong models from weak ones? A benchmark
# everyone scores the same on is not useful regardless of how hard it is.
spread = full.groupby("dataset").accuracy.agg(["min", "max", "std"])
spread["range"] = spread["max"] - spread["min"]
print(spread.round(1).to_string())

In [ ]:
# One figure per pillar. 4a takes the top two rows (a-e), 4b the third (f-g), 4c the
# fourth (h). Height is close to Nature's 247 mm ceiling, which is the trade for keeping
# three sub-pillars in one numbered figure.
fig = plt.figure(figsize=ns.mm(ns.DOUBLE_COL_MM, 268))
outer = fig.add_gridspec(3, 1, height_ratios=[2.05, 0.74, 1.00], hspace=0.44)
grid = outer[0].subgridspec(2, 3, height_ratios=[1, 0.88], hspace=0.72, wspace=0.62)

# --- a: MCQ accuracy ---
ax = fig.add_subplot(grid[0, 0])
x = np.arange(len(MCQ_DATASETS))
width = 0.13
for index, model in enumerate(ns.MODEL_ORDER):
    subset = full[full.model == model].set_index("dataset").reindex(MCQ_DATASETS)
    ax.bar(x + (index - 2.5) * width, subset.accuracy, width,
           color=ns.MODEL_COLORS[model], label=ns.MODEL_LABELS[model])
for index, dataset in enumerate(MCQ_DATASETS):
    chance = full[full.dataset == dataset].chance.iloc[0]
    ax.plot([index - 0.42, index + 0.42], [chance, chance], ls="--", lw=0.6, color="#666666")
ax.set_xticks(x); ax.set_xticklabels(MCQ_DATASETS, fontsize=7, rotation=30,
                                     ha="right", rotation_mode="anchor")
ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 78)
ax.set_title("MCQ", fontsize=7, pad=2)
ns.panel_label_below(ax, "a")

# --- close-ended accuracy ---
ax = fig.add_subplot(grid[0, 1])
x = np.arange(len(CE_DATASETS))
for index, model in enumerate(ns.MODEL_ORDER):
    subset = close_ended[close_ended.model == model].set_index("dataset").reindex(CE_DATASETS)
    ax.bar(x + (index - 2.5) * width, subset.accuracy, width, color=ns.MODEL_COLORS[model])
ax.axhline(50, ls="--", lw=0.6, color="#666666")
ax.text(2.45, 51.5, "chance", fontsize=6.5, color="#666666")
ax.set_xticks(x)
ax.set_xticklabels([d.replace("_CE", "") for d in CE_DATASETS], fontsize=7,
                   rotation=30, ha="right", rotation_mode="anchor")
ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 90)
ax.set_title("Close-ended (yes/no)", fontsize=7, pad=2)
ns.panel_label_below(ax, "b")

# --- open-ended BERTScore, baseline-rescaled ---
ax = fig.add_subplot(grid[0, 2])
x = np.arange(len(OE_DATASETS))
for index, model in enumerate(ns.MODEL_ORDER):
    subset = (open_ended[open_ended.model == model]
              .set_index("dataset").reindex(OE_DATASETS))
    ax.bar(x + (index - 2.5) * width, subset.bertscore_f1_rescaled, width,
           color=ns.MODEL_COLORS[model])
# Rescaled BERTScore has a meaningful zero: "no better than unrelated text". The raw
# score's ~85 floor hides that PathVQA sits at or below it.
ax.axhline(0, lw=0.5, color="black")
ax.set_xticks(x)
ax.set_xticklabels([d.replace("_OE", "") for d in OE_DATASETS], fontsize=7,
                   rotation=30, ha="right", rotation_mode="anchor")
ax.set_ylabel("BERTScore F1 (rescaled)"); ax.set_ylim(-9, 42)
ax.set_title("Open-ended", fontsize=7, pad=2)
ns.panel_label_below(ax, "c")

# --- d: the difficulty ranking, raw vs chance-corrected ---
ax = fig.add_subplot(grid[1, :2])
order = difficulty.index.tolist()
y = np.arange(len(order))
ax.barh(y - 0.21, difficulty.accuracy, 0.30, color="#BFBFBF", label="Raw accuracy")
# One colour per MEASURE, not per dataset: the y-axis row labels already identify the
# datasets, so colouring by dataset was redundant and left the legend showing a single
# swatch for three different bar colours.
ax.barh(y + 0.21, difficulty.above_chance, 0.30,
        color=ns.OKABE_ITO["blue"], label="Above chance")
for index, dataset in enumerate(order):
    ax.text(difficulty.accuracy[dataset] + 0.7, index - 0.21,
            f"{difficulty.accuracy[dataset]:.1f}", va="center", fontsize=6.5)
    ax.text(difficulty.above_chance[dataset] + 0.7, index + 0.21,
            f"{difficulty.above_chance[dataset]:.1f}", va="center", fontsize=6.5,
            fontweight="bold")
ax.set_yticks(y); ax.set_yticklabels(order, fontsize=7)
# Room below the last bar for the pointer note; without it the note sits on the bars.
ax.set_ylim(len(order) - 0.5, -1.55)
ax.set_xlabel("Mean over six models (%)")
ax.set_xlim(0, 58)
# Inside the axes, upper right: d's bars are anchored at x=0 and none reaches the
# right edge, so this corner is free. Keeping the key here rather than above the panel
# saves a whole legend row in a block that has none to spare.
ax.legend(loc="upper right", fontsize=6.5, frameon=False,
          handlelength=1.2, handletextpad=0.4, borderpad=0.1)
# Short pointer only - the full statement is in the caption. Every longer form collided
# with a neighbouring panel's letter or with d's own legend: d's strip is not tall enough
# to carry a two-line sentence beside a legend.
ax.text(0.99, 0.015, "chance-corrected ranking differs",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=6, color="#B33A3A")
ns.panel_label_below(ax, "d")

# --- e: discrimination range ---
ax = fig.add_subplot(grid[1, 2])
for index, dataset in enumerate(MCQ_DATASETS):
    subset = full[full.dataset == dataset]
    ax.plot([index, index], [subset.accuracy.min(), subset.accuracy.max()],
            lw=3.2, color="#8C8C8C", solid_capstyle="round")
    ax.plot([index] * len(subset), subset.accuracy, "o", ms=3.6,
            color=ns.OKABE_ITO["blue"], mec="white", mew=0.5)
    ax.text(index, subset.accuracy.max() + 2.5,
            f"{subset.accuracy.max() - subset.accuracy.min():.0f}",
            ha="center", fontsize=6.5)
ax.set_xticks(range(len(MCQ_DATASETS)))
ax.set_xticklabels(MCQ_DATASETS, fontsize=7, rotation=30, ha="right",
                   rotation_mode="anchor")
ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 78)
ax.set_title("Model spread", fontsize=7, pad=3)
# Key inside the axes: the lowest datum is ~26 on a 0-78 scale, so the floor of this panel
# is empty. Drawn as a miniature of the marks themselves rather than as words, so the
# reader matches shape to shape.
ax.plot([-0.34, -0.34], [7.0, 13.0], lw=3.2, color="#8C8C8C", solid_capstyle="round",
        clip_on=False)
ax.text(-0.16, 10.0, "range over\n6 models", fontsize=5.8, color="#666666",
        va="center", ha="left")
ax.plot([1.30], [10.0], "o", ms=3.6, color=ns.OKABE_ITO["blue"], mec="white", mew=0.5,
        clip_on=False)
ax.text(1.46, 10.0, "one model", fontsize=5.8, color="#666666", va="center", ha="left")
ns.panel_label_below(ax, "e")

handles = [plt.Rectangle((0, 0), 1, 1, fc=ns.MODEL_COLORS[m]) for m in ns.MODEL_ORDER]
_abc = [a for a in fig.axes if abs(a.get_position().y1 - max(
    x.get_position().y1 for x in fig.axes)) < 1e-6]
_row_top = max(a.get_position().y1 for a in _abc)
_title_top = 0.0
fig.canvas.draw()
_r = fig.canvas.get_renderer()
for a in _abc:
    if a.title.get_text():
        _title_top = max(_title_top, fig.transFigure.inverted().transform(
            (0, a.title.get_window_extent(_r).y1))[1])
model_legend = fig.legend(
    handles, [ns.MODEL_LABELS[m] for m in ns.MODEL_ORDER],
    loc="lower center", bbox_to_anchor=(0.5, max(_row_top, _title_top) + 0.004),
    ncol=6, fontsize=6.5, frameon=False, columnspacing=1.2, handletextpad=0.4)

print("panels a-e drawn")

## Figure 8 — resolution sensitivity (4b)

PathOPEN-only: neither comparator publishes magnification metadata. Kruskal-Wallis rather
than ANOVA because accuracy per bin is a proportion over few items, not a normal variate.

In [ ]:
from scipy.stats import kruskal

MAG_ORDER = sorted(magnification.magnification.unique(),
                   key=lambda m: float(str(m).lower().replace("x", "")))
counts = magnification[magnification.model == magnification.model.iloc[0]] \
    .set_index("magnification")["n"].reindex(MAG_ORDER)
print(counts.to_string())

# Per model: is accuracy independent of magnification? Reconstruct binary correct/incorrect
# vectors from the per-bin accuracies, which is what a rank test needs.
trend_rows = []
for model in ns.MODEL_ORDER:
    subset = magnification[magnification.model == model].set_index("magnification").reindex(MAG_ORDER)
    groups = []
    for mag in MAG_ORDER:
        n = int(subset.loc[mag, "n"])
        correct = int(round(n * subset.loc[mag, "accuracy"] / 100))
        groups.append(np.array([1] * correct + [0] * (n - correct)))
    stat, p = kruskal(*groups)
    trend_rows.append({"model": model, "H": round(stat, 2), "p": round(p, 4),
                       "significant": p < 0.05})
magnification_trend = pd.DataFrame(trend_rows)
print()
print(magnification_trend.to_string(index=False))

In [ ]:
from matplotlib.transforms import Bbox

sub = outer[1].subgridspec(1, 2, width_ratios=[1.4, 1], wspace=0.62)
axes = [fig.add_subplot(sub[0, 0]), fig.add_subplot(sub[0, 1])]

# --- a: accuracy vs magnification, one line per model ---
ax = axes[0]
x = np.arange(len(MAG_ORDER))
for model in ns.MODEL_ORDER:
    subset = magnification[magnification.model == model].set_index("magnification").reindex(MAG_ORDER)
    ax.plot(x, subset.accuracy, marker="o", ms=2.8, lw=0.9,
            color=ns.MODEL_COLORS[model], label=ns.MODEL_LABELS[model])
ax.set_xticks(x)
ax.set_xticklabels(MAG_ORDER, fontsize=7)
ax.set_xlabel("Magnification", fontsize=7)
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 82)
# n per bin, reddened below the paper's own n>=30 threshold. Three of six bins are thin,
# and the two extremes (50x, 400x) are the thinnest - which is where the eye is drawn.
for index, mag in enumerate(MAG_ORDER):
    n = int(counts[mag])
    ax.text(index, 2, f"n={n}", ha="center", fontsize=6,
            color="#666666" if n >= MIN_BIN_N else "#B33A3A")
ns.panel_label_below(ax, "f")

# --- b: is the variation significant? ---
ax = axes[1]
y = np.arange(len(ns.MODEL_ORDER))
significant = magnification_trend.set_index("model").reindex(ns.MODEL_ORDER)
ax.barh(y, significant.H, 0.55,
        color=[ns.MODEL_COLORS[m] for m in ns.MODEL_ORDER])
for index, model in enumerate(ns.MODEL_ORDER):
    p = significant.loc[model, "p"]
    ax.text(significant.loc[model, "H"] + 0.25, index,
            ns.significance_stars(p), va="center", fontsize=6.5,
            color="black" if p < 0.05 else "#666666")
ax.set_yticks(y)
ax.set_yticklabels([ns.MODEL_LABELS[m] for m in ns.MODEL_ORDER], fontsize=6.5)
ax.set_xlabel("Kruskal-Wallis $H$", fontsize=7)
ax.invert_yaxis()
n_significant = int(significant.significant.sum())
ax.set_xlim(right=ax.get_xlim()[1] * 1.95)
ax.text(0.98, 0.02, f"{n_significant}/6 models vary\nsignificantly with\nmagnification",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=6.5, color="#666666")
ns.panel_label_below(ax, "g")

# One legend spanning f and g: both encode the same six models in the same colours, so a
# key on f alone under-claims - it looks as though g's bars were unlabelled.
fig.canvas.draw()
_r = fig.canvas.get_renderer()
_fg = Bbox.union([axes[0].get_window_extent(_r), axes[1].get_window_extent(_r)])
_handles = [plt.Line2D([], [], color=ns.MODEL_COLORS[m], lw=2.2, label=ns.MODEL_LABELS[m])
            for m in ns.MODEL_ORDER]
fg_legend = fig.legend(
    _handles, [h.get_label() for h in _handles], loc="lower center",
    bbox_to_anchor=((_fg.x0 + _fg.x1) / 2 / fig.get_window_extent().width,
                    max(axes[0].get_position().y1, axes[1].get_position().y1) + 0.004),
    ncol=6, frameon=False, fontsize=6.5, columnspacing=1.2, handletextpad=0.4)
ns.lift_legend_above_titles(fig, fg_legend, [a for a in axes if a.title.get_text()])

print("panels f-g drawn")

## Figure 9 — organ stratification (4c)

Pooled to organ **system**. The raw metadata has 42 organ labels over 229 items — 21 of
them hold 1–4 items, and five would show "100% accuracy" off a single question. Pooling
to the top-level system (organs are named `System - Subsite`) gives 11 bins; those under
n=10 are grouped as *Other*.

Even pooled, only Gastrointestinal (n=44) clears n≥30, so every row carries its *n*.

In [ ]:
pivot = organ_system.pivot(index="organ_system", columns="model", values="accuracy")
sizes = (organ_system[organ_system.model == organ_system.model.iloc[0]]
         .set_index("organ_system")["n"])
pivot = pivot.reindex(sizes.sort_values(ascending=False).index)[ns.MODEL_ORDER]

ax = fig.add_subplot(outer[2].subgridspec(1, 3, width_ratios=[0.14, 1, 0.14])[0, 1])
image = ax.imshow(pivot.values, cmap="RdYlBu", vmin=0, vmax=90, aspect="auto")

for row in range(pivot.shape[0]):
    for column in range(pivot.shape[1]):
        value = pivot.values[row, column]
        ax.text(column, row, f"{value:.0f}", ha="center", va="center", fontsize=6.5,
                color="white" if value < 25 or value > 75 else "black")

ax.set_xticks(range(len(ns.MODEL_ORDER)))
ax.set_xticklabels([ns.MODEL_LABELS[m] for m in ns.MODEL_ORDER],
                   rotation=30, ha="right", rotation_mode="anchor", fontsize=7)
ax.set_yticks(range(len(pivot)))
# n in the row label, reddened below threshold - the alternative is a reader assuming a
# 13-item row is as reliable as a 44-item one.
ax.set_yticklabels([f"{organ}  (n={int(sizes[organ])})" for organ in pivot.index],
                   fontsize=7)
for tick, organ in zip(ax.get_yticklabels(), pivot.index):
    if sizes[organ] < MIN_BIN_N:
        tick.set_color("#B33A3A")

colorbar = fig.colorbar(image, ax=ax, fraction=0.022, pad=0.015)
colorbar.set_label("Accuracy (%)", fontsize=7)
colorbar.ax.tick_params(labelsize=5)
ax.set_title("PathOPEN MCQ accuracy by organ system", fontsize=7.5, pad=4)
# The "n < 30" convention is stated in the caption rather than on the figure: it cost a
# full text row under a panel that needs the height for its 11 organ rows.
for spine in ax.spines.values():
    spine.set_visible(False)

ns.panel_label_below(ax, "h")
paths = ns.save(fig, "fig04_pillar4_benchmarking")
print("wrote:", paths)
plt.show()

## Draft captions

---

**Fig. 7 | Cross-dataset difficulty across six vision-language models.**
**a**, Zero-shot MCQ accuracy on the three multiple-choice datasets; dashed lines give
per-dataset chance level, which differs because option counts range from 2 to 16.
**b**, Close-ended (yes/no) accuracy; chance is 50%.
**c**, Open-ended answer quality, BERTScore F1 rescaled against the empirical baseline so
that 0 denotes similarity no greater than unrelated text. Filtered PathVQA sits at or
below zero for four of six models, reflecting its 7.8-word mean reference length.
**d**, Datasets ranked by difficulty, showing raw accuracy and accuracy above chance.
Corrected for chance, PathOPEN (22.4 points above chance) lies between PathMMU (25.7) and
PatchVQA (20.5). Metrics are compared **within** format only, never pooled across
formats.
**e**, Range of model accuracies per dataset, as a measure of how well each benchmark
separates strong from weak models.

---

**Fig. 8 | Accuracy by image magnification (PathOPEN only).**
**a**, MCQ accuracy grouped by the magnification metadata PathOPEN records for every
image; no comparator dataset publishes this, so the analysis is possible on PathOPEN
alone. Sample size is given per bin and shown in red where it falls below the n = 30
flag threshold (20×, 50×, 400×).
**b**, Kruskal-Wallis test of whether accuracy depends on magnification, per model.
A non-parametric test is used because per-bin accuracy is a proportion over few items.

---

**Fig. 9 | MCQ accuracy by organ system (PathOPEN only).**
Cells give accuracy per model and organ system. PathOPEN's raw metadata distinguishes 42
organ labels across 229 items, of which 21 hold four or fewer; labels are therefore
pooled to the top-level organ system, and systems with fewer than ten items are grouped
as *Other*. Sample size appears in each row label, in red where below the n = 30
threshold — only Gastrointestinal (n = 44) clears it. No comparator dataset carries organ
metadata, so this stratification is unique to PathOPEN.

---

### Numbers a caption must not get wrong

| quantity | value |
|---|---|
| above-chance difficulty (PathMMU / PathOPEN / PatchVQA) | 25.7 / 22.4 / 20.5 |
| raw accuracy | 50.1 / 42.4 / 41.4 |
| chance | 24.3 / 20.0 / 20.9 |
| PathVQA_OE rescaled BERTScore | −5.8 to 7.4 (four models ≤ 1.4) |
| magnification bins below n=30 | 20×, 50×, 400× (3 of 6) |
| organ systems below n=30 | 10 of 11 |
| raw organ labels pooled | 42 → 11 |

### The claim that must be revised

§2.4 states PathOPEN is *"a harder, more discriminating benchmark for current VLMs than
existing patch-level pathology datasets."* **The first half is not supported.** Corrected
for chance, PathOPEN is mid-range: 22.4 points above chance against PathMMU's 25.7. Raw
accuracy makes it appear harder than PathMMU only because its fixed 5-option format sets
chance at 20% rather than 24.3%.

A reviewer can compute this table from the reported numbers, so the paper should say it
first. Suggested revision:

> PathOPEN presents difficulty comparable to existing patch-level benchmarks while
> additionally enabling resolution- and organ-stratified evaluation that no comparator
> dataset supports.

That is still a real contribution — Figures 8 and 9 exist only because PathOPEN carries
magnification and organ metadata — and it is defensible against the table above.

Table 2 should be updated to match: Pillar 4's claim column currently reads "Harder
benchmark; enables stratified analysis".

**Panel h row labels.** Organ systems are ordered by item count, given in each row label. Labels printed in red mark systems with *n* < 30 — below the threshold this paper uses to flag a stratum as thinly sampled — so those rows should not be read as reliable per-organ estimates. (Stated here rather than on the figure, where the note cost a full text row.)
